<a href="https://colab.research.google.com/github/taiwoas1/Rahman-Taiwo-Database-and-Analytics/blob/main/MONGODB_OPTIMISATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/[USERNAME]/northstar-analytics-cw1/blob/main/notebooks/05_mongodb_optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5 — MongoDB Atlas: Index and Query optimisation

Rahman Taiwo Aina-Saliu NorthStar Case Study — Database and Analytics

Build indexes on service cases and check for performance

In [2]:
!pip install pymongo dnspython -q

from pymongo import MongoClient, ASCENDING, DESCENDING
from datetime import datetime
from google.colab import userdata

MONGO_URI = userdata.get('MONGO_URI')
client = MongoClient(MONGO_URI)
db = client['northstar']
cases = db['service_cases']

print(f'Cases in collection: {cases.count_documents({})}')
print(f'Existing indexes:')
for idx in cases.list_indexes(): print(f'  {idx["name"]}: {idx["key"]}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 17.4 MB/s eta 0:00:00
Cases in collection: 886
Existing indexes:
  _id_: SON([('_id', 1)])


## 1. Baseline — query without any custom index

In [3]:

for idx in cases.list_indexes():
    if idx['name'] != '_id_':
        cases.drop_index(idx['name'])


baseline = db.command('explain',
    {'find': 'service_cases',
     'filter': {'route.pickup_zone': 'Central', 'delivery.status': 'Failed'}},
    verbosity='executionStats')

plan = baseline['queryPlanner']['winningPlan']


if 'executionStats' in baseline:
    stats = baseline['executionStats']
    docs_examined = stats.get('totalDocsExamined', 'N/A')
    docs_returned = stats.get('totalDocsReturned', 'N/A')
    exec_ms = stats.get('executionTimeMillis', 'N/A')
else:
    docs_examined = 'N/A'
    docs_returned = 'N/A'
    exec_ms = 'N/A'

print(f'Stage            : {plan["stage"]}')
print(f'totalDocsExamined: {docs_examined}')
print(f'totalDocsReturned: {docs_returned}')
print(f'executionMillis  : {exec_ms}')

baseline_docs_examined = docs_examined
baseline_ms = exec_ms

Stage            : COLLSCAN
totalDocsExamined: 886
totalDocsReturned: N/A
executionMillis  : 0


 Baseline expectation: **COLLSCAN** 886 AND 30

## 2. Build the index plan

In [4]:
# Index Plan 1 - compound:zone and status
cases.create_index([('route.pickup_zone', ASCENDING),
                    ('delivery.status', ASCENDING)],
                   name='idx_zone_status', background=True)

# Index Plan 2 - lookup customers
cases.create_index([('customer.customer_id', ASCENDING)],
                   name='idx_customer_id', background=True)

# Index Plan 3 - service segment analytics
cases.create_index([('service_type', ASCENDING)],
                   name='idx_service_type', background=True)

# Index Plan 4 - driver scorecards
cases.create_index([('delivery.driver_ref', ASCENDING)],
                   name='idx_driver_ref', background=True)

# Index Plan 5 - multikey on incidents severity for array
cases.create_index([('incidents.severity', ASCENDING)],
                   name='idx_inc_severity', background=True)

# Index Plan 6 - recent-cases dashboards
cases.create_index([('created_at', DESCENDING)],
                   name='idx_created_desc', background=True)

# Index Plan 7 - covered query for rating dashboards
cases.create_index(
    [('route.pickup_zone', ASCENDING),
     ('delivery.status', ASCENDING),
     ('delivery.customer_rating', ASCENDING)],
    name='idx_zone_status_rating_cov', background=True)

print('Built indexes:')
for idx in cases.list_indexes():
    print(f'  {idx["name"]}: {dict(idx["key"])}')

Built indexes:
  _id_: {'_id': 1}
  idx_zone_status: {'route.pickup_zone': 1, 'delivery.status': 1}
  idx_customer_id: {'customer.customer_id': 1}
  idx_service_type: {'service_type': 1}
  idx_driver_ref: {'delivery.driver_ref': 1}
  idx_inc_severity: {'incidents.severity': 1}
  idx_created_desc: {'created_at': -1}
  idx_zone_status_rating_cov: {'route.pickup_zone': 1, 'delivery.status': 1, 'delivery.customer_rating': 1}


## 3. Post-index measurement

In [5]:

optimised = db.command('explain',
    {'find': 'service_cases',
     'filter': {'route.pickup_zone': 'Central', 'delivery.status': 'Failed'},
     'projection': {'delivery.customer_rating': 1, '_id': 0}},
    verbosity='executionStats')

plan2 = optimised['queryPlanner']['winningPlan']

def find_index_name(plan_node):
    if 'indexName' in plan_node:
        return plan_node['indexName']
    if 'inputStage' in plan_node:
        return find_index_name(plan_node['inputStage'])
    return None

index_used = find_index_name(plan2)
docs_examined2 = optimised.get('executionStats', {}).get('totalDocsExamined', 0)
exec_ms2 = optimised.get('executionStats', {}).get('executionTimeMillis', 0)

print(f'Stage            : {plan2["stage"]}')
print(f'Index used       : {index_used}')
print(f'totalDocsExamined: {docs_examined2}')
print(f'executionMillis  : {exec_ms2}')
print()
print(f'Improvement summary:')
print(f'  docs examined : {baseline_docs_examined} → {docs_examined2}')
print(f'  exec time (ms): {baseline_ms} → {exec_ms2}')
print()
print("Covered query confirmed: '_id: 0' removes FETCH stage.")
print("MongoDB returns results directly from index - main collection not touched.")

Stage            : PROJECTION_DEFAULT
Index used       : idx_zone_status_rating_cov
totalDocsExamined: 0
executionMillis  : 2

Improvement summary:
  docs examined : 886 → 0
  exec time (ms): 0 → 2

Covered query confirmed: '_id: 0' removes FETCH stage.
MongoDB returns results directly from index - main collection not touched.


## 4. Covered query — fastest possible read pattern

 MongoBD dont fetch documents, it returns results from index

In [6]:
# Covered query explain with command ()
covered = db.command('explain',
    {'find': 'service_cases',
     'filter': {'route.pickup_zone': 'Central', 'delivery.status': 'Failed'},
     'projection': {'delivery.customer_rating': 1, '_id': 0}},
    verbosity='executionStats')

plan3 = covered['queryPlanner']['winningPlan']

def walk(node, indent=0):
    print('  ' * indent + node['stage'] +
          (f' (index: {node.get("indexName")})' if node.get('indexName') else ''))
    if 'inputStage' in node:
        walk(node['inputStage'], indent + 1)

print('Stage chain:')
walk(plan3)
docs3 = covered.get('executionStats', {}).get('totalDocsExamined', 'N/A')
print(f'\ntotalDocsExamined: {docs3}')

Stage chain:
PROJECTION_DEFAULT
  IXSCAN (index: idx_zone_status_rating_cov)

totalDocsExamined: 0


## 5. Aggregation pipeline optimisation

Three patterns: $match early, $project to drop unused fields, $sort backed by an index.

In [7]:
from datetime import datetime

optimised_pipeline = [
    {'$match': {'delivery.status': {'$in': ['Failed', 'Delayed']},
                'created_at': {'$gte': datetime(2025, 1, 1)}}},
    {'$project': {
        'route.pickup_zone': 1,
        'delivery.status': 1,
        'delivery.customer_rating': 1
    }},
    {'$group': {
        '_id': '$route.pickup_zone',
        'n_problem': {'$sum': 1},
        'avg_rating': {'$avg': '$delivery.customer_rating'}
    }},
    {'$sort': {'n_problem': -1}}
]

explain = db.command('aggregate', 'service_cases',
                     pipeline=optimised_pipeline,
                     explain=True)

for d in cases.aggregate(optimised_pipeline):
    print(d)

## 6. Index utilisation report

Per-index access statistics from MongoDB's $indexStats

In [8]:
for stat in cases.aggregate([{'$indexStats': {}}]):
    print(f"{stat['name']:35s}  ops: {stat['accesses']['ops']:6d}"
          f"  since: {stat['accesses']['since']}")

_id_                                 ops:      2  since: 2026-05-12 01:40:44.357000
idx_zone_status_rating_cov           ops:      0  since: 2026-05-12 21:05:30.948000
idx_service_type                     ops:      0  since: 2026-05-12 21:05:30.202000
idx_customer_id                      ops:      0  since: 2026-05-12 21:05:30.011000
idx_driver_ref                       ops:      0  since: 2026-05-12 21:05:30.392000
idx_created_desc                     ops:      2  since: 2026-05-12 21:05:30.765000
idx_zone_status                      ops:      0  since: 2026-05-12 21:05:29.801000
idx_inc_severity                     ops:      0  since: 2026-05-12 21:05:30.580000
